In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Affichage plus propre des DataFrames
pd.set_option("display.float_format", "{:.2f}".format)

print("Librairies chargées ✓")

Librairies chargées ✓


In [2]:
# Chargement des fichiers qu'on a créés
prices       = pd.read_csv("../data/prices.csv",
                            index_col=0, parse_dates=True)
fundamentals = pd.read_csv("../data/fundamentals.csv",
                            index_col=0)
tickers      = pd.read_csv("../data/jse_tickers.csv")
dataset      = pd.read_csv("../data/dataset.csv",
                            parse_dates=["date"])

print(f"Prix         : {prices.shape}")
print(f"Fondamentaux : {fundamentals.shape}")
print(f"Dataset ML   : {dataset.shape}")
print(f"\nPériode : {prices.index[0].date()} → {prices.index[-1].date()}")

Prix         : (1249, 22)
Fondamentaux : (22, 9)
Dataset ML   : (24533, 16)

Période : 2019-01-02 → 2023-12-29


In [3]:
# On normalise tous les prix à 100 au départ
# pour comparer les performances facilement
prices_norm = prices / prices.iloc[0] * 100

fig = px.line(
    prices_norm,
    title="Performance des 22 actions JSE (base 100)",
    labels={"value": "Performance (base 100)", "Date": "Date"},
    template="plotly_white",
    width=900, height=500
)

fig.update_layout(
    legend_title="Ticker",
    hovermode="x unified"
)

fig.show()

In [4]:
# La corrélation montre quelles actions bougent ensemble
# Utile pour comprendre la diversification du portefeuille
returns_daily = prices.pct_change().dropna()
corr_matrix   = returns_daily.corr()

fig = px.imshow(
    corr_matrix,
    title="Corrélation entre les actions JSE",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    width=800, height=700,
    text_auto=".2f"
)

fig.update_layout(template="plotly_white")
fig.show()

In [5]:
# On visualise la distribution de chaque ratio financier
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[
        "P/E Ratio", "Price/Book", "Debt/Equity",
        "Return on Equity", "Return on Assets",
        "Profit Margin", "Current Ratio", "Float Shares"
    ]
)

cols_to_plot = [
    "trailingPE", "priceToBook", "debtToEquity",
    "returnOnEquity", "returnOnAssets",
    "profitMargins", "currentRatio", "floatShares"
]

positions = [
    (1,1), (1,2), (1,3), (1,4),
    (2,1), (2,2), (2,3), (2,4)
]

for col, (row, col_pos) in zip(cols_to_plot, positions):
    fig.add_trace(
        go.Box(
            y=fundamentals[col],
            name=col,
            showlegend=False,
            marker_color="#1D9E75"
        ),
        row=row, col=col_pos
    )

fig.update_layout(
    title="Distribution des ratios financiers — JSE",
    template="plotly_white",
    height=600, width=900
)

fig.show()

In [6]:
# On visualise le déséquilibre entre les deux classes
label_counts = dataset["label"].value_counts().reset_index()
label_counts.columns = ["label", "count"]
label_counts["label"] = label_counts["label"].map(
    {0: "Sous-performe", 1: "Surperforme"}
)

fig = px.pie(
    label_counts,
    values="count",
    names="label",
    title="Distribution du label de surperformance",
    color_discrete_map={
        "Surperforme"   : "#1D9E75",
        "Sous-performe" : "#D85A30"
    },
    template="plotly_white",
    width=500, height=400
)

fig.show()

print(f"Taux de surperformance : {dataset['label'].mean():.1%}")

Taux de surperformance : 33.7%


In [7]:
# Quelles actions ont le mieux performé sur la période ?
total_returns = (prices.iloc[-1] / prices.iloc[0] - 1) * 100
total_returns = total_returns.sort_values(ascending=True)

fig = px.bar(
    x=total_returns.values,
    y=total_returns.index,
    orientation="h",
    title="Performance totale des actions JSE (2019-2024)",
    labels={"x": "Rendement total (%)", "y": "Ticker"},
    color=total_returns.values,
    color_continuous_scale=["#D85A30", "#F5C4B3", "#1D9E75"],
    template="plotly_white",
    width=700, height=600
)

fig.update_layout(coloraxis_showscale=False)
fig.show()